문제0

In [72]:
import pandas as pd

df = pd.read_csv('data_NHTS2016_complete.csv')

df['O_GU'] = df['P_Depart_code'].astype(str).str[:5]
df['D_GU'] = df['P_Arrival_code'].astype(str).str[:5]

# 내부통행 제거
df = df[df['O_GU'] != df['D_GU']].copy()

print(len(df))

69689


문제1

In [18]:
# ===== 통행목적 분류 (student + P_Trip_purpose 조합) =====
def purpose_new(row):
    if row['P_Trip_purpose'] == 0:        # 출근/등교
        return 'school' if row['student'] == 1 else 'work'
    elif row['P_Trip_purpose'] == 1:      # 업무/학원
        return 'academy' if row['student'] == 1 else 'business'
    else:                                 # 2,3,4 (여가/외식/귀가)
        return 'other'

df['purpose_new'] = df.apply(purpose_new, axis=1)

# ===== 3대 목적으로 통합 =====
def purpose_final(x):
    if x in ['work', 'business']:
        return 'work_business'
    elif x in ['school', 'academy']:
        return 'school_academy'
    else:
        return 'other'

df['purpose_final'] = df['purpose_new'].apply(purpose_final)
print(df['purpose_final'].value_counts())

# ===== 종속변수: 구별 발생량(Production) =====
production = pd.pivot_table(
    df, index='O_GU', columns='purpose_final',
    values='ID', aggfunc='count', fill_value=0
)

# ===== 종속변수: 구별 유입량(Attraction) =====
attraction = pd.pivot_table(
    df, index='D_GU', columns='purpose_final',
    values='ID', aggfunc='count', fill_value=0
)

print(production)
print(attraction)

purpose_final
other             39346
work_business     28540
school_academy     1803
Name: count, dtype: int64
purpose_final  other  school_academy  work_business
O_GU                                               
11110           3026              79            856
11140           2934              57            749
11170           1266              58            904
11200           1387              91           1058
11215           1273              77           1306
11230           1600              57           1025
11260            667              81           1197
11290           1561             136           1386
11305            820              74           1205
11320            621              92           1282
11350           1199              36           1258
11380            700              80           1388
11410           1548              80           1159
11440           2091              57           1014
11470            901              97           1186
1150

In [9]:
import statsmodels.api as sm

In [35]:
# ============================================================
# 문제 1. Trip Generation Model — (1.2) Multiple Linear Regression
# ============================================================
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ---------- [문제 0] 데이터 로드 & 존 내부통행 제거 ----------
df = pd.read_csv('data_NHTS2016_complete.csv')
df['O_GU'] = df['P_Depart_code'].astype(str).str[:5]
df['D_GU'] = df['P_Arrival_code'].astype(str).str[:5]
df = df[df['O_GU'] != df['D_GU']].copy()
print('내부통행 제거 후 통행 수:', len(df))

# ---------- [종속변수] 통행목적 분류 & 구별 발생/유입 집계 ----------
def purpose_new(row):
    if   row['P_Trip_purpose'] == 0: return 'school'  if row['student']==1 else 'work'
    elif row['P_Trip_purpose'] == 1: return 'academy' if row['student']==1 else 'business'
    else:                            return 'other'
df['purpose_new'] = df.apply(purpose_new, axis=1)

def purpose_final(x):
    if   x in ['work','business']:   return 'work_business'
    elif x in ['school','academy']:  return 'school_academy'
    else:                            return 'other'
df['purpose_final'] = df['purpose_new'].apply(purpose_final)
print(df['purpose_final'].value_counts())

production  = pd.pivot_table(df, index='O_GU', columns='purpose_final',
                             values='ID', aggfunc='count', fill_value=0)
attraction  = pd.pivot_table(df, index='D_GU', columns='purpose_final',
                             values='ID', aggfunc='count', fill_value=0)

# ---------- [독립변수] 사회경제지표 구 단위 집계 ----------
# 통계청 7자리코드 -> 구 5자리코드 매핑
code = pd.read_excel('2016 행정구역 코드.xlsx')
code = code.dropna(subset=[code.columns[5], code.columns[6]])
code['ZONE_ID'] = code.iloc[:,5].astype(int)
code['GU']      = code.iloc[:,6].astype('int64').astype(str).str[:5]
zone2gu = dict(zip(code['ZONE_ID'], code['GU']))

def read_txt(fname, ncols):
    d = pd.read_csv(fname, sep='\t', header=None, skiprows=1, encoding='cp949')
    return d.iloc[:, :ncols]

def to_gu(d):                               # 행정동 -> 구 (서울만)
    d = d.copy()
    d['GU'] = d['ZONE_ID'].map(zone2gu)
    d = d.dropna(subset=['GU'])
    d = d[d['GU'].str.startswith('11')]
    return d.drop(columns=['ZONE_ID']).groupby('GU').sum()

def build_X(year):                          # 연도별 독립변수 생성 (16/30 공용)
    sub  = read_txt(f'SUB_POP{year}.TXT', 204)
    work = read_txt(f'WORK_POP{year}.TXT', 4)
    emp  = read_txt(f'EMP_POP{year}.TXT', 204)
    stu  = read_txt(f'STU_POP{year}.TXT', 7)

    sub_df = pd.DataFrame({'ZONE_ID': sub.iloc[:,1].astype(int)})
    sub_df['총인구']      = sub.iloc[:,2:204].sum(axis=1)
    sub_df['고령자']      = sub.iloc[:,2+65:2+101].sum(axis=1) + sub.iloc[:,2+101+65:2+202].sum(axis=1)
    sub_df['생산가능인구'] = sub.iloc[:,2+15:2+65].sum(axis=1) + sub.iloc[:,2+101+15:2+101+65].sum(axis=1)

    work_df = pd.DataFrame({'ZONE_ID': work.iloc[:,1].astype(int),
                            '종사자_3차': work.iloc[:,2], '종사자_총': work.iloc[:,3]})
    emp_df  = pd.DataFrame({'ZONE_ID': emp.iloc[:,1].astype(int)})
    emp_df['취업자_총'] = emp.iloc[:,2:204].sum(axis=1)
    stu_df  = pd.DataFrame({'ZONE_ID': stu.iloc[:,1].astype(int),
                            '초':stu.iloc[:,2],'중':stu.iloc[:,3],'고':stu.iloc[:,4],
                            '특수':stu.iloc[:,5],'대':stu.iloc[:,6]})
    stu_df['총학생'] = stu_df[['초','중','고','특수','대']].sum(axis=1)

    Xy = to_gu(sub_df).join([to_gu(work_df), to_gu(emp_df), to_gu(stu_df)])
    # 파생변수 (다중공선성 완화용)
    Xy['고령자비율']  = Xy['고령자'] / Xy['총인구']
    Xy['종사자_비3차'] = Xy['종사자_총'] - Xy['종사자_3차']
    Xy['취업률']     = Xy['취업자_총'] / Xy['생산가능인구']
    Xy['학생비율']   = Xy['총학생'] / Xy['총인구']
    return Xy

X = build_X(16)
print('독립변수 X shape:', X.shape)

# ---------- [회귀] 6개 모형 (전부 VIF<10 & 모든 변수 p<0.05) ----------
P = production.copy(); P.index = P.index.astype(str); dfP = P.join(X)
A = attraction.copy(); A.index = A.index.astype(str); dfA = A.join(X)

def vif_max(xvars, data):
    Xm = sm.add_constant(data[list(xvars)])
    return max(variance_inflation_factor(Xm.values, i) for i in range(1, Xm.shape[1]))

final = {}
def fit(name, y, xvars, data):
    m = sm.OLS(data[y], sm.add_constant(data[xvars])).fit()
    final[name] = (y, xvars, m)
    print(f"\n{'='*55}\n[{name}] R²={m.rsquared:.3f}  adjR²={m.rsquared_adj:.3f}  maxVIF={vif_max(xvars,data):.2f}")
    for v in m.params.index:
        print(f"  {v:12s} coef={m.params[v]:12.4f}  p={m.pvalues[v]:.4f}")

fit('P_work',  'work_business',  ['취업자_총','고령자비율','취업률'],   dfP)  # ①출근/업무 발생
fit('A_work',  'work_business',  ['종사자_3차','종사자_비3차','총인구'], dfA)  # ②출근/업무 유입
fit('P_school','school_academy', ['고','학생비율','총인구'],            dfP)  # ③등교/학원 발생
fit('A_school','school_academy', ['고','대'],                          dfA)  # ④등교/학원 유입
fit('P_other', 'other',          ['총인구','종사자_3차','종사자_비3차'], dfP)  # ⑤나머지 발생
fit('A_other', 'other',          ['총인구','취업률'],                   dfA)  # ⑥나머지 유입

내부통행 제거 후 통행 수: 69689
purpose_final
other             39346
work_business     28540
school_academy     1803
Name: count, dtype: int64
독립변수 X shape: (25, 16)

[P_work] R²=0.628  adjR²=0.575  maxVIF=1.31
  const        coef=   6082.7420  p=0.0045
  취업자_총        coef=      0.0024  p=0.0001
  고령자비율        coef=   4751.7027  p=0.0215
  취업률          coef=  -8416.2037  p=0.0054

[A_work] R²=0.899  adjR²=0.884  maxVIF=3.23
  const        coef=    859.3261  p=0.0003
  종사자_3차       coef=      0.0067  p=0.0000
  종사자_비3차      coef=     -0.0129  p=0.0174
  총인구          coef=     -0.0019  p=0.0011

[P_school] R²=0.273  adjR²=0.169  maxVIF=2.80
  const        coef=     31.1471  p=0.1866
  고            coef=     -0.0043  p=0.0237
  학생비율         coef=    124.8185  p=0.0484
  총인구          coef=      0.0002  p=0.0292

[A_school] R²=0.609  adjR²=0.574  maxVIF=1.04
  const        coef=     12.5545  p=0.3160
  고            coef=      0.0048  p=0.0007
  대            coef=      0.0010  p=0.0001

[P_other] R²=

In [62]:
# ============================================================
# 문제 1. (1.3) 2030년 구별 통행 발생/유입 예측
# ============================================================
X30 = build_X(30)                              # 2030년 독립변수 (셀1의 build_X 재사용)
print('X30 shape:', X30.shape)

def predict(name, data):
    y, xvars, m = final[name]
    return m.predict(sm.add_constant(data[xvars], has_constant='add'))

pred = pd.DataFrame(index=X30.index)
pred['work_P']   = predict('P_work',   X30)    # 출근/업무 발생
pred['work_A']   = predict('A_work',   X30)    # 출근/업무 유입
pred['school_P'] = predict('P_school', X30)    # 등교/학원 발생
pred['school_A'] = predict('A_school', X30)    # 등교/학원 유입
pred['other_P']  = predict('P_other',  X30)    # 나머지 발생
pred['other_A']  = predict('A_other',  X30)    # 나머지 유입
pred = pred.clip(lower=0)                       # 음수 예측 방지

# 전 목적 합산 = 2030년 구별 발생/유입 총량 (문제 2 입력값)
pred['O_total'] = pred[['work_P','school_P','other_P']].sum(axis=1)
pred['D_total'] = pred[['work_A','school_A','other_A']].sum(axis=1)

print(pred.round(0))
print('\n[2030 발생 합계]', pred['O_total'].sum().round(0))
print('[2030 유입 합계]', pred['D_total'].sum().round(0))

# ------------------------------------------------------------
# 엑셀 파일 저장 코드 추가 (기본 서식)
# ------------------------------------------------------------
pred.round(0).to_excel('(1-3)_2030년_구별_통행발생_유입_예측결과.xlsx', index_label='자치구')

X30 shape: (25, 16)
       work_P  work_A  school_P  school_A  other_P  other_A  O_total  D_total
GU                                                                           
11110   702.0  1780.0      77.0      84.0   2220.0    711.0   3000.0   2575.0
11140   539.0  2633.0      64.0      60.0   3195.0    495.0   3797.0   3187.0
11170   743.0  1313.0      60.0      47.0   1762.0    857.0   2565.0   2218.0
11200   841.0  1077.0      78.0      60.0   1460.0    881.0   2379.0   2019.0
11215   903.0   753.0      76.0      78.0   1137.0    976.0   2115.0   1807.0
11230   910.0   860.0      88.0     105.0   1258.0    895.0   2255.0   1860.0
11260  1005.0   487.0      71.0      43.0    829.0    769.0   1905.0   1299.0
11290  1183.0   673.0      95.0     122.0   1079.0   1253.0   2357.0   2048.0
11305  1007.0   705.0      61.0      36.0   1084.0    711.0   2152.0   1452.0
11320  1081.0   705.0      61.0      42.0   1091.0    807.0   2233.0   1554.0
11350  1397.0   715.0      64.0     117.0   

In [37]:
# ============================================================
# 문제 2. Trip Distribution Model — 기준 데이터 준비
# ============================================================
gus = sorted(production.index.astype(str))      # 25개 구 코드

# ----- 2016 기준 OD 행렬 (전 목적 합산) -----
od_base = pd.pivot_table(df, index='O_GU', columns='D_GU',
                         values='ID', aggfunc='count', fill_value=0)
od_base = od_base.reindex(index=gus, columns=gus, fill_value=0).astype(float)

# ----- 기준연도(2016) marginal: 가구통행실태조사 따름 -----
O_base = od_base.sum(axis=1)     # 행합 = 발생(유출)
D_base = od_base.sum(axis=0)     # 열합 = 유입

# ----- 장래(2030) marginal: (1.3) 예측값 따름 -----
O_future = pred['O_total'].reindex(gus)
D_future = pred['D_total'].reindex(gus)

print('기준 OD 총통행:', od_base.values.sum())
print('2030 발생합계:', O_future.sum().round(0), '/ 2030 유입합계:', D_future.sum().round(0))

# ----- 불균형 정합화: 유입(D)을 발생(O) 총합에 맞춰 비례 스케일링 -----
D_future = D_future * (O_future.sum() / D_future.sum())
print('정합화 후 2030 유입합계:', D_future.sum().round(0))   # = 발생합계와 동일

기준 OD 총통행: 69689.0
2030 발생합계: 69290.0 / 2030 유입합계: 58074.0
정합화 후 2030 유입합계: 69290.0


In [38]:
# ============================================================
# (2.1) Average Growth Factor
# ============================================================
# 성장인자
F_i = (O_future / O_base).reindex(gus)     # 발생측 성장인자
F_j = (D_future / D_base).reindex(gus)     # 유입측 성장인자

print("===== 발생측 성장인자 F_i =====")
print(F_i.round(3))
print("\n===== 유입측 성장인자 F_j =====")
print(F_j.round(3))

# Average Growth Factor 적용: T_ij = t_ij * (F_i + F_j)/2
avg_factor = (F_i.values[:, None] + F_j.values[None, :]) / 2   # 25x25 평균인자 행렬
T_avg = od_base.values * avg_factor

T_avg = pd.DataFrame(T_avg, index=gus, columns=gus)
print("\n===== Average Growth Factor OD 행렬 (일부) =====")
print(T_avg.iloc[:5, :5].round(1))
print("\n총통행:", T_avg.values.sum().round(0))

===== 발생측 성장인자 F_i =====
GU
11110    0.757
11140    1.015
11170    1.151
11200    0.938
11215    0.796
11230    0.841
11260    0.980
11290    0.765
11305    1.025
11320    1.119
11350    1.048
11380    1.060
11410    0.866
11440    1.010
11470    1.163
11500    1.524
11530    1.060
11545    1.481
11560    0.980
11590    0.727
11620    0.770
11650    1.080
11680    1.059
11710    0.936
11740    1.372
dtype: float64

===== 유입측 성장인자 F_j =====
GU
11110    0.776
11140    1.015
11170    1.191
11200    0.945
11215    0.816
11230    0.833
11260    0.797
11290    0.791
11305    0.824
11320    0.930
11350    1.063
11380    0.989
11410    0.893
11440    1.094
11470    1.220
11500    1.514
11530    0.978
11545    1.297
11560    0.992
11590    0.708
11620    0.702
11650    1.221
11680    1.166
11710    1.000
11740    1.246
dtype: float64

===== Average Growth Factor OD 행렬 (일부) =====
       11110  11140  11170  11200  11215
11110    0.0  369.6  159.8  130.2   86.5
11140  368.0    0.0  360.8  282.3  

In [39]:
# Average Growth Factor의 marginal 오차 확인
print("발생 marginal 오차 (행합 - 목표):")
print((T_avg.sum(axis=1) - O_future).round(1).head())
print("\n유입 marginal 오차 (열합 - 목표):")
print((T_avg.sum(axis=0) - D_future).round(1).head())

발생 marginal 오차 (행합 - 목표):
11110    412.5
11140    -91.6
11170   -182.2
11200     41.8
11215    274.0
dtype: float64

유입 marginal 오차 (열합 - 목표):
11110    404.0
11140    -88.4
11170   -240.0
11200     23.8
11215    247.9
dtype: float64


In [68]:
# ============================================================
# (2.2) Frata Model (회차별 OD 행렬 및 성장인자 증명 추출 버전)
# ============================================================
import numpy as np
import pandas as pd

T = od_base.values.copy().astype(float)     # 2016 기준 OD에서 시작
O_t = O_future.reindex(gus).values           # 목표 발생
D_t = D_future.reindex(gus).values           # 목표 유입

n_iter = 7        # 5회 이상 수행 조건 만족 (7회)
fratar_history = {} # 회차별 결과를 저장할 딕셔너리

for it in range(1, n_iter + 1):
    # --- 행(발생) 보정 ---
    row_sum = T.sum(axis=1)
    Fi = np.divide(O_t, row_sum, out=np.ones_like(O_t), where=row_sum!=0)
    T = T * Fi[:, None]
    
    # --- 열(유입) 보정 ---
    col_sum = T.sum(axis=0)
    Fj = np.divide(D_t, col_sum, out=np.ones_like(D_t), where=col_sum!=0)
    T = T * Fj[None, :]

    # --- 수렴 확인용 성장인자 출력 ---
    print(f"[Iter {it}] Fi 범위 {Fi.min():.4f}~{Fi.max():.4f} | "
          f"Fj 범위 {Fj.min():.4f}~{Fj.max():.4f}")
    
    # --- [추가] 과제 증명용 데이터 구조화 (OD 행렬 + Fi + Fj 결합) ---
    # 1) 현재 회차의 25x25 OD 행렬 생성
    df_iter = pd.DataFrame(T.copy(), index=gus, columns=gus)
    # 2) 우측 끝에 해당 회차의 행 성장인자(Fi) 컬럼 추가
    df_iter['Growth_Factor_Fi'] = Fi
    # 3) 하단 끝에 해당 회차의 열 성장인자(Fj) 로우 추가 (우측 끝 공백은 NaN 처리)
    fj_row = list(Fj) + [np.nan]
    df_iter.loc['Growth_Factor_Fj'] = fj_row
    
    # 딕셔너리에 회차별 통합 데이터프레임 저장
    fratar_history[it] = df_iter

# 최종 수렴된 결과는 기존과 동일하게 저장 (이후 문제 연동용)
T_frata = pd.DataFrame(T, index=gus, columns=gus)

# ============================================================
# [추가] 각 Iteration 과정 전체를 시트별로 분리하여 엑셀 파일 추출
# ============================================================
excel_filename = '(2-2)_Fratar_모든_Iteration_과정증명.xlsx'
with pd.ExcelWriter(excel_filename) as writer:
    for it in range(1, n_iter + 1):
        # 보기 편하게 소수점 첫째자리까지 반올림하여 엑셀 시트로 저장
        fratar_history[it].round(1).to_excel(writer, sheet_name=f'Iteration_{it}')

print(f"\n✅ [증명 완료] 1회차부터 {n_iter}회차까지의 OD 행렬과 성장인자가 '{excel_filename}' 파일로 저장되었습니다.")

print("\n===== Frata 결과 (일부) =====")
print(T_frata.iloc[:5, :5].round(1))
print("\n총통행:", T_frata.values.sum().round(0))
print("발생 marginal 오차 최대:", np.abs(T_frata.sum(axis=1) - O_future).max().round(3))
print("유입 marginal 오차 최대:", np.abs(T_frata.sum(axis=0) - D_future).max().round(3))

[Iter 1] Fi 범위 0.7273~1.5236 | Fj 범위 0.6917~1.4641
[Iter 2] Fi 범위 0.8922~1.0690 | Fj 범위 0.9698~1.0372
[Iter 3] Fi 범위 0.9839~1.0129 | Fj 범위 0.9935~1.0062
[Iter 4] Fi 범위 0.9966~1.0030 | Fj 범위 0.9984~1.0015
[Iter 5] Fi 범위 0.9991~1.0008 | Fj 범위 0.9996~1.0004
[Iter 6] Fi 범위 0.9997~1.0002 | Fj 범위 0.9999~1.0001
[Iter 7] Fi 범위 0.9999~1.0001 | Fj 범위 1.0000~1.0000

✅ [증명 완료] 1회차부터 7회차까지의 OD 행렬과 성장인자가 '(2-2)_Fratar_모든_Iteration_과정증명.xlsx' 파일로 저장되었습니다.

===== Frata 결과 (일부) =====
       11110  11140  11170  11200  11215
11110    0.0  335.8  155.0  115.5   69.0
11140  332.8    0.0  415.4  292.1  107.1
11170  147.8  402.3    0.0   94.8   47.9
11200  113.8  286.2   92.2    0.0  311.2
11215   65.7  100.9   50.6  314.1    0.0

총통행: 69290.0
발생 marginal 오차 최대: 0.058
유입 marginal 오차 최대: 0.0


In [41]:
import numpy as np

# ============================================================
# (2.3) Singly Constrained Gravity Model
# ============================================================
# ----- 1) Separation C_ij: 존간 승용차·대중교통 통행시간 평균 -----
df['M_Trip_mode_4'].value_counts()   # 먼저 수단 값 확인

M_Trip_mode_4
PT      43016
Auto    21571
Walk     3327
Bike      896
Taxi      879
Name: count, dtype: int64

In [42]:
# ----- 1) Separation C_ij: 존간 승용차(Auto)·대중교통(PT) 통행시간 평균 -----
mask = df['M_Trip_mode_4'].isin(['PT', 'Auto'])
cij_long = df[mask].groupby(['O_GU', 'D_GU'])['tr_time'].mean()

# 25x25 행렬로 변환
C = cij_long.unstack().reindex(index=gus, columns=gus)

# 결측(해당 OD에 PT/Auto 통행이 없던 경우) 처리
print("C_ij 결측 셀 수:", C.isna().sum().sum())
print("C_ij 평균 통행시간:", np.nanmean(C.values).round(2))
print(C.iloc[:5, :5].round(1))

C_ij 결측 셀 수: 25
C_ij 평균 통행시간: 44.08
D_GU   11110  11140  11170  11200  11215
O_GU                                    
11110    NaN   22.6   27.3   30.8   37.5
11140   22.7    NaN   25.5   24.4   32.0
11170   27.5   24.4    NaN   28.6   38.6
11200   29.0   23.1   28.9    NaN   20.9
11215   37.5   31.6   39.2   20.6    NaN


In [43]:
# ----- 2) β 보정: 2016 관측 평균통행시간을 재현하는 β 탐색 -----
# 2016 기준 OD(od_base)의 관측 평균 통행시간 (C_ij 가중평균)
Cv = C.values.copy()
obs = od_base.values.copy()
valid = ~np.isnan(Cv)                      # 대각선 제외
mean_cost_obs = (obs[valid] * Cv[valid]).sum() / obs[valid].sum()
print("2016 관측 평균통행시간:", round(mean_cost_obs, 3))

# β를 바꿔가며, 모형이 만드는 평균통행시간이 관측과 같아지는 값 탐색
def model_mean_cost(beta, O, D):
    Cm = np.where(valid, Cv, 0.0)
    f = np.where(valid, np.exp(-beta * Cm), 0.0)   # 마찰함수
    A = 1.0 / (f * D[None, :]).sum(axis=1)          # 발생제약 정규화
    T = A[:, None] * O[:, None] * D[None, :] * f
    return (T * Cm).sum() / T.sum(), T

# 이분탐색으로 β 보정 (2016 marginal 사용)
O16, D16 = od_base.sum(axis=1).values, od_base.sum(axis=0).values
lo, hi = 0.0001, 1.0
for _ in range(60):
    mid = (lo + hi) / 2
    mc, _ = model_mean_cost(mid, O16, D16)
    if mc > mean_cost_obs:   # 모형 평균비용이 크면 β를 키워 더 줄여야
        lo = mid
    else:
        hi = mid
beta = (lo + hi) / 2
print("보정된 β:", round(beta, 5))
print("β 적용시 모형 평균통행시간:", round(model_mean_cost(beta, O16, D16)[0], 3))

2016 관측 평균통행시간: 34.239
보정된 β: 0.08124
β 적용시 모형 평균통행시간: 34.239


In [74]:
# ----- 3) Singly Constrained Gravity Model 적용 (2030) -----
O = O_future.reindex(gus).values
D = D_future.reindex(gus).values

Cm = np.where(valid, Cv, 0.0)
f  = np.where(valid, np.exp(-beta * Cm), 0.0)        # 마찰함수 exp(-βC_ij)

A = 1.0 / (f * D[None, :]).sum(axis=1)                # 발생제약 정규화 A_i
T_singly = A[:, None] * O[:, None] * D[None, :] * f   # T_ij = A_i O_i D_j f_ij

T_singly = pd.DataFrame(T_singly, index=gus, columns=gus)

print("===== Singly Constrained Gravity 결과 (일부) =====")
print(T_singly.iloc[:5, :5].round(1))
print("\n총통행:", T_singly.values.sum().round(0))

# 제약 만족 여부 확인
print("\n[발생 제약] 행합 - 목표 O_i  (만족해야 함):")
print((T_singly.sum(axis=1) - O_future).abs().max().round(4))
print("[유입 제약] 열합 - 목표 D_j  (만족 안 함 = Singly의 특징):")
print((T_singly.sum(axis=0) - D_future).abs().max().round(2))

# ----- [추가 코드] Singly Constrained Gravity 결과 엑셀 출력 -----
file_name = "(2-3)_Singly_Gravity_결과.xlsx"
T_singly.to_excel(file_name)
print(f"\n[성공] '{file_name}' 파일이 저장되었습니다.")

===== Singly Constrained Gravity 결과 (일부) =====
       11110  11140  11170  11200  11215
11110    0.0  497.0  235.9  161.3   83.9
11140  495.3    0.0  340.8  338.9  163.0
11170  202.1  323.3    0.0  145.8   57.8
11200  162.6  324.1  140.1    0.0  219.4
11215   79.8  159.4   60.0  247.0    0.0

총통행: 69290.0

[발생 제약] 행합 - 목표 O_i  (만족해야 함):
0.0
[유입 제약] 열합 - 목표 D_j  (만족 안 함 = Singly의 특징):
1222.62

[성공] '(2-3)_Singly_Gravity_결과.xlsx' 파일이 저장되었습니다.


In [75]:
# ============================================================
# (2.4) Doubly Constrained Gravity Model
# ============================================================
O = O_future.reindex(gus).values
D = D_future.reindex(gus).values
f = np.where(valid, np.exp(-beta * Cm), 0.0)     # 동일한 β, 마찰함수

A = np.ones(len(gus))
B = np.ones(len(gus))      # B_j = 1로 시작

doubly_history = {}
n_iter = 7
for it in range(1, n_iter + 1):
    # A_i 갱신 (B 고정)
    A = 1.0 / (B[None, :] * D[None, :] * f).sum(axis=1)
    # B_j 갱신 (갱신된 A 사용)
    B = 1.0 / (A[:, None] * O[:, None] * f).sum(axis=0)

    T = A[:, None] * O[:, None] * B[None, :] * D[None, :] * f
    doubly_history[it] = pd.DataFrame(T.copy(), index=gus, columns=gus)

    row_err = np.abs(T.sum(axis=1) - O).max()
    col_err = np.abs(T.sum(axis=0) - D).max()
    print(f"[Iter {it}] 발생오차 {row_err:8.3f} | 유입오차 {col_err:8.3f} | 총통행 {T.sum():.0f}")

T_doubly = doubly_history[n_iter]
print("\n===== Doubly Constrained 결과 (일부) =====")
print(T_doubly.iloc[:5, :5].round(1))

# ◀ [추가] (2.4) Doubly Constrained 반복 과정 엑셀 파일로 통째로 추출
with pd.ExcelWriter('(2-4)_Doubly_Gravity_모든_Iteration_과정증명.xlsx') as writer:
    for it in range(1, n_iter + 1):
        if it in doubly_history:
            # 기존 doubly_history에 담긴 행렬을 엑셀 시트별로 저장
            pd.DataFrame(doubly_history[it], index=gus, columns=gus).round(1).to_excel(writer, sheet_name=f'Iteration_{it}')

print("✅ (2.4) Doubly Gravity 회차별 OD 행렬 추출 완료!")

# ----- [추가 코드] Doubly Constrained Gravity 최종 결과 엑셀 출력 -----
file_name = "(2-4)_Doubly_Gravity_최종결과.xlsx"
T_doubly.to_excel(file_name)
print(f"\n[성공] '{file_name}' 파일이 저장되었습니다.")

[Iter 1] 발생오차  209.891 | 유입오차    0.000 | 총통행 69290
[Iter 2] 발생오차   27.556 | 유입오차    0.000 | 총통행 69290
[Iter 3] 발생오차    3.898 | 유입오차    0.000 | 총통행 69290
[Iter 4] 발생오차    0.576 | 유입오차    0.000 | 총통행 69290
[Iter 5] 발생오차    0.122 | 유입오차    0.000 | 총통행 69290
[Iter 6] 발생오차    0.031 | 유입오차    0.000 | 총통행 69290
[Iter 7] 발생오차    0.008 | 유입오차    0.000 | 총통행 69290

===== Doubly Constrained 결과 (일부) =====
       11110  11140  11170  11200  11215
11110    0.0  397.1  181.0  116.2   65.9
11140  398.0    0.0  267.2  249.5  130.8
11170  161.4  262.3    0.0  106.7   46.1
11200  128.4  260.1  108.0    0.0  173.1
11215   60.7  123.1   44.5  172.1    0.0
✅ (2.4) Doubly Gravity 회차별 OD 행렬 추출 완료!

[성공] '(2-4)_Doubly_Gravity_최종결과.xlsx' 파일이 저장되었습니다.


In [67]:
## ============================================================
# (2.5) Average / Frata / Doubly Gravity 결과 비교
# ============================================================
methods = {'Average': T_avg, 'Frata': T_frata, 'Doubly_Gravity': T_doubly}

# 1) 제약 만족도 비교
print("===== 제약 만족도 (목표 marginal과의 최대 오차) =====")
for name, M in methods.items():
    re = (M.sum(axis=1) - O_future).abs().max()
    ce = (M.sum(axis=0) - D_future).abs().max()
    print(f"  {name:15s} 발생오차 {re:8.2f} | 유입오차 {ce:8.2f}")

# 2) 평균 통행시간 비교 (거리저항 반영 여부)
print("\n===== 모형별 평균 통행시간 =====")
for name, M in methods.items():
    v = M.values; mc = (v[valid]*Cv[valid]).sum()/v[valid].sum()
    print(f"  {name:15s} {mc:.2f} 분")
print(f"  (2016 관측)      {mean_cost_obs:.2f} 분")

# 3) OD별 차이: 같은 셀을 세 방법이 얼마나 다르게 추정하는지
print("\n===== 방법 간 OD 차이 (Frata 대비) =====")
diff_avg = (T_avg - T_frata).abs().values[valid].mean()
diff_dbl = (T_doubly - T_frata).abs().values[valid].mean()
print(f"  |Average - Frata| 평균: {diff_avg:.2f}")
print(f"  |Doubly  - Frata| 평균: {diff_dbl:.2f}")

===== 제약 만족도 (목표 marginal과의 최대 오차) =====
  Average         발생오차   476.49 | 유입오차   508.41
  Frata           발생오차     0.06 | 유입오차     0.00
  Doubly_Gravity  발생오차     0.01 | 유입오차     0.00

===== 모형별 평균 통행시간 =====
  Average         34.41 분
  Frata           34.61 분
  Doubly_Gravity  34.90 분
  (2016 관측)      34.24 분

===== 방법 간 OD 차이 (Frata 대비) =====
  |Average - Frata| 평균: 12.64
  |Doubly  - Frata| 평균: 27.97


In [47]:
print(df[['P_Depart_x','P_Depart_y']].describe())

         P_Depart_x    P_Depart_y
count  69689.000000  69689.000000
mean     126.993004     37.547853
std        0.075818      0.049546
min      126.806086     37.440830
25%      126.927800     37.505820
50%      127.003111     37.547301
75%      127.048800     37.578710
max      127.174800     37.680460


In [48]:
# ----- 구별 Centroid (구 내 모든 통행 출발·도착 좌표의 평균) -----
# 출발지·도착지 좌표를 모두 모아 구별 중심 계산 (더 안정적)
dep = df[['O_GU','P_Depart_x','P_Depart_y']].rename(
        columns={'O_GU':'GU','P_Depart_x':'x','P_Depart_y':'y'})
arr = df[['D_GU','P_Arrival_x','P_Arrival_y']].rename(
        columns={'D_GU':'GU','P_Arrival_x':'x','P_Arrival_y':'y'})
coords = pd.concat([dep, arr])
centroid = coords.groupby('GU')[['x','y']].mean().reindex(gus)

print(centroid.round(5))

               x         y
GU                        
11110  126.99133  37.57763
11140  126.99302  37.56166
11170  126.97583  37.53664
11200  127.04151  37.55256
11215  127.08317  37.54627
11230  127.05261  37.58228
11260  127.08746  37.59574
11290  127.02558  37.60043
11305  127.02399  37.63132
11320  127.03844  37.65677
11350  127.06735  37.64450
11380  126.91966  37.60692
11410  126.93899  37.57281
11440  126.92816  37.55544
11470  126.85829  37.52678
11500  126.84360  37.55127
11530  126.87604  37.49555
11545  126.89688  37.46636
11560  126.90998  37.51710
11590  126.94737  37.49711
11620  126.93782  37.47829
11650  127.01375  37.48797
11680  127.04708  37.50426
11710  127.11069  37.50718
11740  127.13790  37.54320


In [49]:
import requests, time

import os
KAKAO_KEY = os.environ["KAKAO_REST_API_KEY"]  # set your own key as an environment variable
headers = {"Authorization": f"KakaoAK {KAKAO_KEY}"}
URL = "https://apis-navi.kakaomobility.com/v1/directions"

# 자동차 경로 시간(분)·거리(km) 행렬
auto_time = pd.DataFrame(np.nan, index=gus, columns=gus)   # 분
auto_dist = pd.DataFrame(np.nan, index=gus, columns=gus)   # km

for i in gus:
    ox, oy = centroid.loc[i, 'x'], centroid.loc[i, 'y']
    for j in gus:
        if i == j:
            continue
        dx, dy = centroid.loc[j, 'x'], centroid.loc[j, 'y']
        params = {"origin": f"{ox},{oy}", "destination": f"{dx},{dy}"}
        try:
            r = requests.get(URL, headers=headers, params=params, timeout=10)
            summary = r.json()['routes'][0]['summary']
            auto_time.loc[i, j] = summary['duration'] / 60      # 초→분
            auto_dist.loc[i, j] = summary['distance'] / 1000    # m→km
        except Exception as e:
            print(f"실패 {i}->{j}: {e}")
        time.sleep(0.05)   # 호출 간 약간 대기
    print(f"{i} 완료")

print("\n자동차 통행시간(분) 일부:")
print(auto_time.iloc[:5, :5].round(1))
print("\n자동차 거리(km) 일부:")
print(auto_dist.iloc[:5, :5].round(1))

11110 완료
11140 완료
11170 완료
11200 완료
11215 완료
11230 완료
11260 완료
11290 완료
11305 완료
11320 완료
11350 완료
11380 완료
11410 완료
11440 완료
11470 완료
11500 완료
11530 완료
11545 완료
11560 완료
11590 완료
11620 완료
11650 완료
11680 완료
11710 완료
11740 완료

자동차 통행시간(분) 일부:
       11110  11140  11170  11200  11215
11110    NaN   18.4   25.0   39.8   54.1
11140   17.8    NaN   17.9   27.5   45.5
11170   31.3   19.0    NaN   34.0   47.8
11200   31.5   26.9   30.0    NaN   17.2
11215   37.4   35.8   39.7   14.2    NaN

자동차 거리(km) 일부:
       11110  11140  11170  11200  11215
11110    NaN    3.1    6.3    7.5   12.5
11140    3.4    NaN    5.8    5.5    9.8
11170    8.0    4.5    NaN    9.9   15.6
11200    8.9    7.3   11.1    NaN    9.6
11215   10.1    9.0   11.7    4.4    NaN


In [50]:
# ============================================================
# (3.1) Choice set generation — 3개 수단 통행시간/통행비용 추정
# ============================================================
mask_v = ~np.eye(len(gus), dtype=bool)     # 대각선 제외 마스크

# ---------- 통행시간 (분) ----------
tt_auto = auto_time.copy()                                  # Auto+Taxi
tt_pt   = auto_time * 1.4                                    # PT (대중교통 1.4배 가정)
tt_walk = (auto_dist / 9.5) * 60                            # Walk+Bike (평속 9.5km/h)

# ---------- 통행비용 (원) ----------
# Auto+Taxi: 연료비 + 택시비의 평균
fuel = auto_dist * 130                                       # 연료비: km당 130원 (연비12km/L·휘발유1560원)
taxi = 4800 + (auto_dist * 1000 / 131).clip(lower=0) * 100  # 택시: 기본4800원 + 131m당 100원
cost_auto = (fuel + taxi) / 2

# PT: 대중교통 요금 (기본 1500원 + 10km 초과 5km당 100원)
extra = (auto_dist - 10).clip(lower=0)
cost_pt = 1500 + np.ceil(extra / 5) * 100

# Walk+Bike: 무료
cost_walk = auto_dist * 0

print("===== 통행시간(분) 일부 =====")
print("Auto :", tt_auto.iloc[0,1].round(1), "| PT:", tt_pt.iloc[0,1].round(1),
      "| Walk:", tt_walk.iloc[0,1].round(1))
print("\n===== 통행비용(원) 일부 =====")
print("Auto :", cost_auto.iloc[0,1].round(0), "| PT:", cost_pt.iloc[0,1].round(0),
      "| Walk:", cost_walk.iloc[0,1].round(0))

===== 통행시간(분) 일부 =====
Auto : 18.4 | PT: 25.8 | Walk: 19.9

===== 통행비용(원) 일부 =====
Auto : 3806.0 | PT: 1500.0 | Walk: 0.0


In [51]:
# ============================================================
# (3.1) Choice set generation 완성 — 통행별 대안 속성 매핑
# ============================================================
# ----- 실제 선택 수단을 3개로 집계 (M_Trip_mode_4 -> 3 modes) -----
def mode3(x):
    if x == 'PT':                  return 'PT'
    elif x in ['Walk', 'Bike']:    return 'WalkBike'
    elif x in ['Auto', 'Taxi']:    return 'AutoTaxi'
    return None
df['chosen'] = df['M_Trip_mode_4'].apply(mode3)

# ----- 행렬을 (O_GU, D_GU) -> 값 딕셔너리로 변환하는 헬퍼 -----
def to_dict(M): return {(i, j): M.loc[i, j] for i in gus for j in gus if i != j}
TT = {'PT': to_dict(tt_pt), 'WalkBike': to_dict(tt_walk), 'AutoTaxi': to_dict(tt_auto)}
CC = {'PT': to_dict(cost_pt), 'WalkBike': to_dict(cost_walk), 'AutoTaxi': to_dict(cost_auto)}

# ----- 각 통행에 3개 대안의 시간·비용 부여 -----
choice = df[['ID','O_GU','D_GU','chosen']].copy()
key = list(zip(choice['O_GU'], choice['D_GU']))
for m in ['PT','WalkBike','AutoTaxi']:
    choice[f'time_{m}'] = [TT[m].get(k, np.nan) for k in key]
    choice[f'cost_{m}'] = [CC[m].get(k, np.nan) for k in key]

# 선택지 속성이 모두 채워진 통행만 사용 (대각선/결측 제거)
choice = choice.dropna().reset_index(drop=True)

print("Choice set 통행 수:", len(choice))
print("\n선택수단 분포:")
print(choice['chosen'].value_counts())
print("\n샘플:")
print(choice.head(3).to_string())

Choice set 통행 수: 69689

선택수단 분포:
chosen
PT          43016
AutoTaxi    22450
WalkBike     4223
Name: count, dtype: int64

샘플:
         ID   O_GU   D_GU chosen     time_PT  cost_PT  time_WalkBike  cost_WalkBike  time_AutoTaxi  cost_AutoTaxi
0  100000_1  11470  11170     PT   44.963333   1600.0      82.408421            0.0      32.116667    8228.272672
1  100000_1  11170  11470     PT   47.296667   1600.0      86.911579            0.0      33.783333    8546.755076
2  100021_1  11590  11320     PT  103.950000   1900.0     186.694737            0.0      74.250000   15603.842748


In [52]:
# (3.1) 결과 검증용 출력 (보고서/제출용)

# 1) 600개 OD 수단별 시간·비용 행렬이 채워졌는지
print("=== 600개 OD 중 채워진 셀 수 (대각선 제외 600개) ===")
print("Auto 시간:", auto_time.notna().sum().sum(), "/ PT 비용:", cost_pt.notna().sum().sum())

# 2) 선택수단별로 3개 대안 평균 속성 (합리성 확인)
print("\n=== 선택수단별 3개 대안 평균 통행시간(분) ===")
print(choice.groupby('chosen')[['time_PT','time_WalkBike','time_AutoTaxi']].mean().round(1))
print("\n=== 선택수단별 3개 대안 평균 통행비용(원) ===")
print(choice.groupby('chosen')[['cost_PT','cost_WalkBike','cost_AutoTaxi']].mean().round(0))

# 3) Choice set 전체 저장 (코드 제출용)
choice.to_csv('choice_set_31.csv', index=False, encoding='utf-8-sig')
print("\nchoice_set_31.csv 저장 완료 — 600 OD × 3수단 속성이 통행별로 매핑됨")

=== 600개 OD 중 채워진 셀 수 (대각선 제외 600개) ===
Auto 시간: 600 / PT 비용: 600

=== 선택수단별 3개 대안 평균 통행시간(분) ===
          time_PT  time_WalkBike  time_AutoTaxi
chosen                                         
AutoTaxi     47.2           68.3           33.7
PT           48.6           69.6           34.7
WalkBike     31.0           38.3           22.1

=== 선택수단별 3개 대안 평균 통행비용(원) ===
          cost_PT  cost_WalkBike  cost_AutoTaxi
chosen                                         
AutoTaxi   1580.0            0.0         7232.0
PT         1583.0            0.0         7325.0
WalkBike   1508.0            0.0         5106.0

choice_set_31.csv 저장 완료 — 600 OD × 3수단 속성이 통행별로 매핑됨


In [55]:
# 차량 보유 여부별 수단 선택 분포
print("차량 보유별 수단선택 비율:")
print(pd.crosstab(df['Home_car'], df['chosen'], normalize='index').round(3))
print("\n운전 가능별 수단선택 비율:")
print(pd.crosstab(df['Home_drive'], df['chosen'], normalize='index').round(3))

차량 보유별 수단선택 비율:
chosen    AutoTaxi     PT  WalkBike
Home_car                           
0            0.052  0.855     0.093
1            0.409  0.541     0.050

운전 가능별 수단선택 비율:
chosen      AutoTaxi     PT  WalkBike
Home_drive                           
0              0.093  0.794     0.114
1              0.404  0.554     0.042


In [59]:
# ============================================================
# (3.2) MNL 최종 — car_WB 제거 (모든 계수 유의)
# ============================================================
def negloglik(p):
    b_time, b_cost = p[0], p[1]
    asc   = np.array([0.0, p[2], p[3]])
    g_car = np.array([0.0, 0.0,  p[4]])    # 차량보유 → AutoTaxi에만
    g_drv = np.array([0.0, p[5], p[6]])    # 운전가능 → WB, AT 둘 다
    V = (b_time*time_mat + b_cost*cost_mat + asc[None,:]
         + g_car[None,:]*car[:,None] + g_drv[None,:]*drive[:,None])
    V = V - V.max(axis=1, keepdims=True)
    expV = np.exp(V)
    P = expV / expV.sum(axis=1, keepdims=True)
    return -np.log(P[np.arange(len(y)), y] + 1e-12).sum()

x0 = [-0.05,-0.1, 0,0, 0, 0,0]
res = minimize(negloglik, x0=x0, method='BFGS')
se = np.sqrt(np.diag(res.hess_inv)); t = res.x/se

labels = ['β_time','β_cost','ASC_WB','ASC_AT','car_AT','drv_WB','drv_AT']
print("===== MNL 최종 추정 (기준: PT) =====")
for l,b,s,tv in zip(labels,res.x,se,t):
    sig = '***' if abs(tv)>2.58 else ('**' if abs(tv)>1.96 else '')
    print(f"  {l:8s} = {b:9.4f}  (t={tv:7.2f}) {sig}")

ll=-res.fun; shares=np.array([(y==k).mean() for k in range(3)]); ll0=np.log(shares)[y].sum()
print(f"\n  LL={ll:.1f} | LL0={ll0:.1f} | McFadden ρ²={1-ll/ll0:.4f}")
b_time, b_cost = res.x[0], res.x[1]

===== MNL 최종 추정 (기준: PT) =====
  β_time   =   -0.0464  (t= -36.95) ***
  β_cost   =   -0.1184  (t= -29.04) ***
  ASC_WB   =   -1.5541  (t=-139.95) ***
  ASC_AT   =   -3.8563  (t= -96.60) ***
  car_AT   =    2.3007  (t=  76.77) ***
  drv_WB   =   -0.5457  (t= -47.04) ***
  drv_AT   =    1.5537  (t=  60.59) ***

  LL=-50112.6 | LL0=-58023.4 | McFadden ρ²=0.1363


In [60]:
# ============================================================
# (3.3) Value of Time (통행시간 가치)
# ============================================================
# β_time: 효용/분, β_cost: 효용/천원 → VoT(원/분) = (β_time / β_cost) * 1000
vot_min  = (b_time / b_cost) * 1000
vot_hour = vot_min * 60

print(f"β_time = {b_time:.4f} (효용/분)")
print(f"β_cost = {b_cost:.4f} (효용/천원)")
print(f"\nVoT = {vot_min:.1f} 원/분")
print(f"VoT = {vot_hour:,.0f} 원/시간")
print(f"\n[참고] 2016 최저임금 6,030원/시간 대비 {vot_hour/6030:.2f}배")

β_time = -0.0464 (효용/분)
β_cost = -0.1184 (효용/천원)

VoT = 392.4 원/분
VoT = 23,541 원/시간

[참고] 2016 최저임금 6,030원/시간 대비 3.90배


In [65]:
# ============================================================
# (3.4) 2030년 수단별 OD 추정
# ============================================================
# ----- 1) OD별 평균 개인속성 (car, drive 보유율) -----
od_attr = ch.groupby(['O_GU','D_GU'])[['Home_car','Home_drive']].mean()

# ----- 2) OD별 수단 분담률 계산 (MNL 적용) -----
PT_share = pd.DataFrame(0.0, index=gus, columns=gus)
WB_share = pd.DataFrame(0.0, index=gus, columns=gus)
AT_share = pd.DataFrame(0.0, index=gus, columns=gus)

for i in gus:
    for j in gus:
        if i == j: continue
        # 해당 OD의 수단별 시간·비용
        tt = np.array([tt_pt.loc[i,j], tt_walk.loc[i,j], tt_auto.loc[i,j]])
        cc = np.array([cost_pt.loc[i,j], cost_walk.loc[i,j], cost_auto.loc[i,j]]) / 1000.0
        # 해당 OD 평균 car/drive (없으면 전체평균)
        if (i,j) in od_attr.index:
            cr, dr = od_attr.loc[(i,j),'Home_car'], od_attr.loc[(i,j),'Home_drive']
        else:
            cr, dr = ch['Home_car'].mean(), ch['Home_drive'].mean()
        # 효용 V (PT 기준)
        asc   = np.array([0.0, res.x[2], res.x[3]])
        g_car = np.array([0.0, 0.0,      res.x[4]])
        g_drv = np.array([0.0, res.x[5], res.x[6]])
        V = b_time*tt + b_cost*cc + asc + g_car*cr + g_drv*dr
        V = V - V.max()
        P = np.exp(V) / np.exp(V).sum()
        PT_share.loc[i,j], WB_share.loc[i,j], AT_share.loc[i,j] = P

# ----- 3) 2030 전체 OD(Doubly) × 수단분담률 -----
OD_PT = T_doubly * PT_share
OD_WB = T_doubly * WB_share
OD_AT = T_doubly * AT_share

print("===== 2030년 수단별 OD 총통행 =====")
print(f"  PT       : {OD_PT.values.sum():,.0f}")
print(f"  WalkBike : {OD_WB.values.sum():,.0f}")
print(f"  AutoTaxi : {OD_AT.values.sum():,.0f}")
print(f"  합계      : {(OD_PT.values+OD_WB.values+OD_AT.values).sum():,.0f}")

# 분담률 (%)
tot = OD_PT.values.sum()+OD_WB.values.sum()+OD_AT.values.sum()
print(f"\n===== 2030 수단분담률 =====")
print(f"  PT {OD_PT.values.sum()/tot*100:.1f}% | WalkBike {OD_WB.values.sum()/tot*100:.1f}% | AutoTaxi {OD_AT.values.sum()/tot*100:.1f}%")
print(f"\n[참고] 2016 실제 분담률: PT 61.7% | WalkBike 6.1% | AutoTaxi 32.2%")

print("\n===== PT OD 일부 =====")
print(OD_PT.iloc[:5,:5].round(1))

# ============================================================
# [추가] 문제 3-4. 2030년 수단별 최종 OD 행렬 엑셀 추출
# ============================================================
with pd.ExcelWriter('(3-4)_2030년_수단별_최종OD_예측.xlsx') as writer:
    OD_PT.round(1).to_excel(writer, sheet_name='대중교통(PT)_OD')
    OD_WB.round(1).to_excel(writer, sheet_name='도보자전거(WB)_OD')
    OD_AT.round(1).to_excel(writer, sheet_name='승용차택시(AT)_OD')
    
print("✅ 문제 3-4 최종 2030년 수단별 OD 엑셀 파일 추출 완료")

===== 2030년 수단별 OD 총통행 =====
  PT       : 47,058
  WalkBike : 4,025
  AutoTaxi : 18,206
  합계      : 69,290

===== 2030 수단분담률 =====
  PT 67.9% | WalkBike 5.8% | AutoTaxi 26.3%

[참고] 2016 실제 분담률: PT 61.7% | WalkBike 6.1% | AutoTaxi 32.2%

===== PT OD 일부 =====
       11110  11140  11170  11200  11215
11110    0.0  255.7  123.9   66.2   38.7
11140  262.1    0.0  195.0  157.5   71.6
11170  111.3  180.7    0.0   73.3   28.1
11200   85.3  180.7   79.0    0.0  132.9
11215   38.5   73.0   25.8  118.3    0.0
✅ 문제 3-4 최종 2030년 수단별 OD 엑셀 파일 추출 완료
